In [1]:
!pip install nltk
!pip install Sastrawi
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import emoji
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import nltk

**BACA DATA**

In [3]:
data_path = 'tiktok 1 4 des 5.30.csv'
df = pd.read_csv(data_path)

In [4]:
df.sample(5)

,link user,views,judul video,content creator,tanggal publish,hastag
18,https://www.tiktok.com/@moms.shiny/video/75788...,507,hanya sir harrison ford berani marahi pak zulh...,moms.shiny,2d ago,NaN
17,https://www.tiktok.com/@soulthflow/video/75790...,115,Mencuat Lagi Vidio Zulhas 2013.,soulthflow,1d ago,NaN
28,https://www.tiktok.com/@arsip_sejarah/video/75...,2977,Flash back klarifikasi zulkifli hasan di sempr...,arsip_sejarah,2d ago,#politik
49,https://www.tiktok.com/@efemer139/video/757777...,32,Kadang orang asing justru lebih lantang membel...,efemer139,5d ago,NaN
42,https://www.tiktok.com/@dutarkarya/video/75791...,58,Pada 2013 Harrison Ford pernah menegur langsun...,dutarkarya,1d ago,#harrisonford


In [5]:
# Mengecek jumlah row & coloumn
print(f'shape: {df.shape}')

shape: (110, 6)


**CLEANING DUPLICATE**

In [6]:
# Cek jumlah nilai duplikat di kolom 'akun'
duplicates = df['judul video'].duplicated(keep=False)  # semua duplikat (termasuk yang pertama)
print(f"Jumlah baris dengan nilai duplikat di 'judul video': {duplicates.sum()}")

# Tampilkan contoh duplikat
print("\nContoh baris dengan nilai duplikat:")
print(df[duplicates].head(10))

Jumlah baris dengan nilai duplikat di 'judul video': 15

Contoh baris dengan nilai duplikat:
                                            link user views judul video  \
21  https://www.tiktok.com/@ima.talks/video/757884...   369         NaN   
22  https://www.tiktok.com/@el_647/video/757913295...    32         NaN   
44  https://www.tiktok.com/@alisadjah/video/757866...   117         NaN   
46  https://www.tiktok.com/@d.pilarienew/video/757...    18         NaN   
48  https://www.tiktok.com/@haay.jull66/video/7578...    22         NaN   
54  https://www.tiktok.com/@momynaybageur/video/75...    27      dengan   
67  https://www.tiktok.com/@ryanmase9/video/757767...    20      dengan   
76  https://www.tiktok.com/@meilandland/video/7578...    35         NaN   
77  https://www.tiktok.com/@najris3/video/75784622...    12         NaN   
79  https://www.tiktok.com/@muhammad.jasril78/vide...    38      dengan   

      content creator tanggal publish               hastag  
21          ima.talk

In [8]:
# Baca data
df = pd.read_csv('tiktok 1 4 des 5.30.csv')

# Hapus duplikat berdasarkan kolom 'akun' — simpan baris pertama
df_clean = df.drop_duplicates(subset=['judul video'], keep='first')

# Tampilkan info
print(f"Sebelum: {len(df)} baris")
print(f"Sesudah: {len(df_clean)} baris")
print(f"Jumlah duplikat dihapus: {len(df) - len(df_clean)}")

# Simpan hasil
df_clean.to_csv('cleaning duplicate tiktok.csv', index=False)

Sebelum: 110 baris
Sesudah: 97 baris
Jumlah duplikat dihapus: 13


**HAPUS TANGGAL SELAIN ---- AGO**

In [9]:
import pandas as pd

# Baca dataset (ganti nama file sesuai kebutuhan)
df = pd.read_csv('tiktok 1 4 des 5.30.csv')  # ← ganti dengan nama file-mu

# Kolom yang berisi waktu (misal: 'waktu', 'tanggal', 'timestamp')
kolom_waktu = 'tanggal publish'  # ← ganti dengan nama kolom yang benar

# Hanya pertahankan baris yang mengandung 'ago' (case-insensitive)
df_filtered = df[df[kolom_waktu].astype(str).str.contains('ago', case=False, na=False)]

# Tampilkan info
print(f"Jumlah baris SEBELUM filter: {len(df)}")
print(f"Jumlah baris SETELAH filter: {len(df_filtered)}")
print(f"Jumlah baris yang dihapus: {len(df) - len(df_filtered)}")

# Simpan hasil
df_filtered.to_csv('data_cleaning_waktu.csv', index=False)

# Tampilkan contoh
print("\nContoh data yang dipertahankan:")
print(df_filtered[kolom_waktu].head(10))

Jumlah baris SEBELUM filter: 110
Jumlah baris SETELAH filter: 105
Jumlah baris yang dihapus: 5

Contoh data yang dipertahankan:
0    4d ago
1    2d ago
2    1d ago
3    2d ago
4    4d ago
5    4d ago
6    4d ago
7    1d ago
8    2d ago
9    3d ago
Name: tanggal publish, dtype: object


**BENERIN TANGGAL PUBLISH**

In [16]:
import pandas as pd
from datetime import datetime, timedelta
import re

# Baca dataset (ganti nama file sesuai kebutuhan)
df = pd.read_csv('data_cleaning_waktu.csv')  # ← GANTI DENGAN NAMA FILEMU

# Kolom yang berisi waktu relatif (misal: 'waktu', 'tanggal posting', dll.)
kolom_waktu = 'tanggal publish'  # ← GANTI DENGAN NAMA KOLOM WAKTU YANG SEBENARNYA

# Tanggal referensi: 4 Desember 2025 pukul 17.30 WIB
reference_datetime = datetime(2025, 12, 3, 17, 30)

def convert_relative_time(time_str):
    if pd.isna(time_str) or not isinstance(time_str, str):
        return 'Invalid Date'

    try:
        s = time_str.strip().lower()
        print(f"Processing: '{s}'")

        # Pola 1: "Xd ago" atau "Xh ago"
        match1 = re.search(r'(\d+)\s*[hd]\s*ago', s)
        # Pola 2: "Xd" atau "Xh" (tanpa 'ago')
        match2 = re.search(r'^(\d+)\s*([hd])$', s)

        if match1:
            number = int(match1.group(1))
            unit = 'd' if 'd' in match1.group(0) else 'h'
        elif match2:
            number = int(match2.group(1))
            unit = match2.group(2)
        else:
            return 'Invalid Date'

        # Hitung tanggal
        if unit == 'h':
            result_dt = reference_datetime - timedelta(hours=number)
        elif unit == 'd':
            result_dt = reference_datetime - timedelta(days=number)
        else:
            return 'Invalid Date'

        # Kembalikan hanya tanggal (format YYYY-MM-DD)
        return result_dt.strftime('%Y-%m-%d')

    except Exception as e:
        print(f"Error converting '{time_str}': {e}")
        return 'Invalid Date'

# Terapkan ke kolom waktu
df['tanggal_absolut'] = df[kolom_waktu].apply(convert_relative_time)

# Opsional: Hapus baris dengan 'Invalid Date'
df_clean = df[df['tanggal_absolut'] != 'Invalid Date'].copy()

# Simpan hasil
df_clean.to_csv('data_waktu_diperbaiki.csv', index=False)

# Tampilkan hasil
print("\n✅ Konversi selesai!")
print(f"Jumlah baris valid: {len(df_clean)} dari {len(df)}")
print("\nContoh hasil:")
print(df_clean.head(10))

Processing: '4d ago'
Processing: '2d ago'
Processing: '1d ago'
Processing: '2d ago'
Processing: '4d ago'
Processing: '4d ago'
Processing: '4d ago'
Processing: '1d ago'
Processing: '2d ago'
Processing: '3d ago'
Processing: '3d ago'
Processing: '4d ago'
Processing: '1d ago'
Processing: '17h ago'
Processing: '3d ago'
Processing: '2d ago'
Processing: '1d ago'
Processing: '1d ago'
Processing: '2d ago'
Processing: '3d ago'
Processing: '2d ago'
Processing: '1d ago'
Processing: '3d ago'
Processing: '2d ago'
Processing: '2d ago'
Processing: '2d ago'
Processing: '3d ago'
Processing: '2d ago'
Processing: '1d ago'
Processing: '19h ago'
Processing: '1d ago'
Processing: '1d ago'
Processing: '4d ago'
Processing: '2d ago'
Processing: '3d ago'
Processing: '1d ago'
Processing: '1d ago'
Processing: '1d ago'
Processing: '1d ago'
Processing: '1d ago'
Processing: '5d ago'
Processing: '1d ago'
Processing: '4d ago'
Processing: '2d ago'
Processing: '2d ago'
Processing: '1d ago'
Processing: '3d ago'
Processing:

**BENERIN JUMLAH LIKE/VIEW**

In [20]:
import pandas as pd
import re

# Baca dataset (ganti nama file sesuai kebutuhan)
df = pd.read_csv('data_waktu_diperbaiki.csv')  # ← GANTI DENGAN NAMA FILEMU

# Kolom yang berisi teks seperti "39.7K", "66.9K", "5185", dll.
kolom_views = 'views'  # ← GANTI DENGAN NAMA KOLOM YANG BENAR

def convert_views(views_str):
    if pd.isna(views_str) or not isinstance(views_str, str):
        return "0"  # atau 0 jika ingin integer

    print(f"Processing: '{views_str}'")

    try:
        s = views_str.strip().lower()

        # Deteksi satuan: K (ribu), M (juta)
        if 'k' in s:
            # Ekstrak angka sebelum 'k'
            match = re.search(r'(\d+\.?\d*)\s*k', s)
            if match:
                number = float(match.group(1)) * 1000
            else:
                # Jika tidak ketemu, coba ekstrak angka apa pun
                num_part = re.sub(r'[^\d.]', '', s)
                if num_part:
                    number = float(num_part) * 1000
                else:
                    return "0"
        elif 'm' in s:
            match = re.search(r'(\d+\.?\d*)\s*m', s)
            if match:
                number = float(match.group(1)) * 1_000_000
            else:
                num_part = re.sub(r'[^\d.]', '', s)
                if num_part:
                    number = float(num_part) * 1_000_000
                else:
                    return "0"
        else:
            # Hanya angka — hapus semua non-digit (kecuali titik)
            num_part = re.sub(r'[^\d.]', '', s)
            if not num_part or num_part == '.':
                return "0"
            number = float(num_part)

        # Kembalikan sebagai string (sesuai permintaanmu)
        return str(int(number))

    except Exception as e:
        print(f"Error converting '{likes_str}': {e}")
        return "0"

# Terapkan ke kolom
df[kolom_views] = df[kolom_views].apply(convert_views)

# Simpan hasil
df.to_csv('data_views_diperbaiki.csv', index=False)

# Tampilkan hasil
print("\n✅ Konversi selesai!")
print(df.head(10))

Processing: '39.7K'
Processing: '66.9K'
Processing: '5185'
Processing: '196'
Processing: '455'
Processing: '103.1K'
Processing: '4621'
Processing: '80'
Processing: '53.5K'
Processing: '2127'
Processing: '185'
Processing: '3730'
Processing: '284'
Processing: '9'
Processing: '321K'
Processing: '339'
Processing: '41'
Processing: '115'
Processing: '507'
Processing: '576'
Processing: '369'
Processing: '32'
Processing: '169'
Processing: '1128'
Processing: '191'
Processing: '17.7K'
Processing: '65'
Processing: '2977'
Processing: '23'
Processing: '39'
Processing: '309'
Processing: '26'
Processing: '300.9K'
Processing: '56'
Processing: '85'
Processing: '67'
Processing: '21'
Processing: '79'
Processing: '71'
Processing: '90'
Processing: '1025'
Processing: '58'
Processing: '150'
Processing: '117'
Processing: '18'
Processing: '55'
Processing: '22'
Processing: '32'
Processing: '30'
Processing: '14'
Processing: '6'
Processing: '7'
Processing: '27'
Processing: '18'
Processing: '31'
Processing: '90'
P

**PREPOCESSING DATA (HAPUS EMOJI & KECILKAN HURUF)**

In [21]:
# Fungsi untuk menghapus emoji
def remove_emoji(text):
    if isinstance(text, str):  # Check if the text is a string before processing
        return emoji.replace_emoji(text, replace='')
    else:
        return text  # Return the original value if it's not a string

# Fungsi untuk membersihkan teks
def clean_text(text):
    # Menghapus emoji
    text = remove_emoji(text)
    # Mengubah ke huruf kecil
    if isinstance(text, str):  # Check if the text is a string before processing
        text = text.lower()
        # Menghapus username, hashtag, dan link
        text = re.sub(r'@\w+|#\w+|http\S+', '', text)
        # Menghapus angka dan tanda baca kecuali huruf
        text = re.sub(r'[^a-z\s]', '', text)
        # Menghapus spasi ekstra
        text = re.sub(r'\s+', ' ', text).strip()
    return text

# Mengaplikasikan fungsi clean_text ke kolom 'comment'
df['cleaned_judul'] = df['judul video'].apply(clean_text)

# Menampilkan data yang sudah dibersihkan
print("\nData Setelah Pembersihan:")
print(df.head())


Data Setelah Pembersihan:
                                           link user  views  \
0  https://www.tiktok.com/@feovles/video/75780292...  39700   
1  https://www.tiktok.com/@dennysumargoreal/video...  66900   
2  https://www.tiktok.com/@metropolitan.id/video/...   5185   
3  https://www.tiktok.com/@25media.anakmuda/photo...    196   
4  https://www.tiktok.com/@si.indo.news/photo/757...    455   

                                         judul video   content creator  \
0  Aktor senior Harrison Ford pernah memarahi Zul...           feovles   
1  APAKAH PAK ZULKIFLI BISA MENJAWAB KERESAHAN RA...  dennysumargoreal   
2  2013, Zulkifli Hasan sempat diwawancara oleh a...   metropolitan.id   
3  Jakarta, Dualima - Setelah sempat viral kembal...  25media.anakmuda   
4      Kadang, alam bukan datang memberi peringatan…      si.indo.news   

  tanggal publish hastag tanggal_absolut  \
0          4d ago    NaN      2025-11-29   
1          2d ago    NaN      2025-12-01   
2          1d ago

In [23]:
# Mengunduh stopwords dan tokenizer
nltk.download('stopwords')
nltk.download('punkt')
import nltk
nltk.download('punkt_tab')

# Menghapus stop words
stop_words = set(stopwords.words('indonesian'))  # Gunakan stopwords bahasa Indonesia
df['cleaned_judul'] = df['cleaned_judul'].apply(
    lambda x: ' '.join([word for word in word_tokenize(x) if word not in stop_words]) if isinstance(x, str) else x
)

# Menampilkan data setelah penghapusan stop words
print("\nData Setelah Penghapusan Stop Words:")
print(df.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



Data Setelah Penghapusan Stop Words:
                                           link user  views  \
0  https://www.tiktok.com/@feovles/video/75780292...  39700   
1  https://www.tiktok.com/@dennysumargoreal/video...  66900   
2  https://www.tiktok.com/@metropolitan.id/video/...   5185   
3  https://www.tiktok.com/@25media.anakmuda/photo...    196   
4  https://www.tiktok.com/@si.indo.news/photo/757...    455   

                                         judul video   content creator  \
0  Aktor senior Harrison Ford pernah memarahi Zul...           feovles   
1  APAKAH PAK ZULKIFLI BISA MENJAWAB KERESAHAN RA...  dennysumargoreal   
2  2013, Zulkifli Hasan sempat diwawancara oleh a...   metropolitan.id   
3  Jakarta, Dualima - Setelah sempat viral kembal...  25media.anakmuda   
4      Kadang, alam bukan datang memberi peringatan…      si.indo.news   

  tanggal publish hastag tanggal_absolut  \
0          4d ago    NaN      2025-11-29   
1          2d ago    NaN      2025-12-01   
2     

In [24]:
# Membuat objek stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Proses stemming
df['stemmed_judul'] = df['cleaned_judul'].apply(
    lambda x: stemmer.stem(x) if isinstance(x, str) else ''
)

# Menampilkan data setelah stemming
print("\nData Setelah Stemming:")
print(df.head())


Data Setelah Stemming:
                                           link user  views  \
0  https://www.tiktok.com/@feovles/video/75780292...  39700   
1  https://www.tiktok.com/@dennysumargoreal/video...  66900   
2  https://www.tiktok.com/@metropolitan.id/video/...   5185   
3  https://www.tiktok.com/@25media.anakmuda/photo...    196   
4  https://www.tiktok.com/@si.indo.news/photo/757...    455   

                                         judul video   content creator  \
0  Aktor senior Harrison Ford pernah memarahi Zul...           feovles   
1  APAKAH PAK ZULKIFLI BISA MENJAWAB KERESAHAN RA...  dennysumargoreal   
2  2013, Zulkifli Hasan sempat diwawancara oleh a...   metropolitan.id   
3  Jakarta, Dualima - Setelah sempat viral kembal...  25media.anakmuda   
4      Kadang, alam bukan datang memberi peringatan…      si.indo.news   

  tanggal publish hastag tanggal_absolut  \
0          4d ago    NaN      2025-11-29   
1          2d ago    NaN      2025-12-01   
2          1d ago   

In [25]:
# Tokenisasi
df['tokens'] = df['stemmed_judul'].apply(
    lambda x: word_tokenize(x) if isinstance(x, str) else []
)

# Menampilkan data setelah tokenisasi
print("\nData Setelah Tokenisasi:")
print(df.head())

# Menyimpan data yang sudah diproses ke file baru
df.to_csv('processed_clean2.csv', index=False)


Data Setelah Tokenisasi:
                                           link user  views  \
0  https://www.tiktok.com/@feovles/video/75780292...  39700   
1  https://www.tiktok.com/@dennysumargoreal/video...  66900   
2  https://www.tiktok.com/@metropolitan.id/video/...   5185   
3  https://www.tiktok.com/@25media.anakmuda/photo...    196   
4  https://www.tiktok.com/@si.indo.news/photo/757...    455   

                                         judul video   content creator  \
0  Aktor senior Harrison Ford pernah memarahi Zul...           feovles   
1  APAKAH PAK ZULKIFLI BISA MENJAWAB KERESAHAN RA...  dennysumargoreal   
2  2013, Zulkifli Hasan sempat diwawancara oleh a...   metropolitan.id   
3  Jakarta, Dualima - Setelah sempat viral kembal...  25media.anakmuda   
4      Kadang, alam bukan datang memberi peringatan…      si.indo.news   

  tanggal publish hastag tanggal_absolut  \
0          4d ago    NaN      2025-11-29   
1          2d ago    NaN      2025-12-01   
2          1d ago 

In [28]:
import pandas as pd

# Baca dataset (ganti dengan nama file-mu)
df = pd.read_csv('processed_clean2.csv')  # ← GANTI SESUAI NAMA FILE

# Hanya pertahankan baris yang mengandung 'ford' di kolom 'judul video'
df_filtered = df[
    df['tokens'].astype(str).str.contains('ford', case=False, na=False)
]

# Tampilkan info
print(f"Jumlah baris SEBELUM filter: {len(df)}")
print(f"Jumlah baris SETELAH filter: {len(df_filtered)}")
print(f"Jumlah baris yang dihapus: {len(df) - len(df_filtered)}")

# Simpan hasil
df_filtered.to_csv('data_hanya_ford.csv', index=False)

# Tampilkan contoh
print("\nContoh judul yang dipertahankan:")
print(df_filtered['judul video'].head(10))

Jumlah baris SEBELUM filter: 105
Jumlah baris SETELAH filter: 58
Jumlah baris yang dihapus: 47

Contoh judul yang dipertahankan:
0     Aktor senior Harrison Ford pernah memarahi Zul...
2     2013, Zulkifli Hasan sempat diwawancara oleh a...
3     Jakarta, Dualima - Setelah sempat viral kembal...
5     Harrison Ford soroti parahnya deforestasi Tess...
7     Pada September 2013, seorang aktor Hollywood b...
9     Zulkifli Hasan Pernah Dimarahi Harrison Ford S...
11    DIAJAK DISKUSI GAK MAU! Zulhas Sebut Harrison ...
14    Menteri kita DIMARAHIN sama Harrison Ford kare...
15    Cuplikan percakapan Harrison Ford dan Menteri ...
16    Tahun 2013, Harrison Ford datang ke Indonesia ...
Name: judul video, dtype: object


In [35]:
import pandas as pd

# Baca dataset yang hanya berisi video "ford"
df = pd.read_csv('data_hanya_ford.csv')

# Pastikan kolom 'tanggal_absolut' dalam format datetime
df['tanggal_absolut'] = pd.to_datetime(df['tanggal_absolut'], errors='coerce')

# Hapus baris dengan tanggal tidak valid (jika ada)
df = df.dropna(subset=['tanggal_absolut'])

# Hitung jumlah video per tanggal
jumlah_per_tanggal = (
    df.groupby('tanggal_absolut')
      .size()
      .reset_index(name='jumlah_video')
)

# Urutkan berdasarkan tanggal
jumlah_per_tanggal = jumlah_per_tanggal.sort_values('tanggal_absolut').reset_index(drop=True)

# Tampilkan hasil
print("📅 Jumlah video Ford per tanggal publish:")
print(jumlah_per_tanggal)

# Simpan ke file baru
jumlah_per_tanggal.to_csv('jumlah_ford_per_tanggal.csv', index=False)

# (Opsional) Tampilkan dengan format tanggal pendek: "29 nov"
jumlah_per_tanggal['tanggal_pendek'] = jumlah_per_tanggal['tanggal_absolut'].dt.strftime('%d %b').str.lower()
print("\nVersi format pendek:")
print(jumlah_per_tanggal[['tanggal_pendek', 'jumlah_video']])

📅 Jumlah video Ford per tanggal publish:
  tanggal_absolut  jumlah_video
0      2025-11-28             1
1      2025-11-29            12
2      2025-11-30            12
3      2025-12-01            15
4      2025-12-02            17
5      2025-12-03             1

Versi format pendek:
  tanggal_pendek  jumlah_video
0         28 nov             1
1         29 nov            12
2         30 nov            12
3         01 dec            15
4         02 dec            17
5         03 dec             1
